In [5]:
import subprocess
import json
import os

In [6]:
pip install surya-ocr


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
def run_surya_ocr_cli(pdf_path, languages, output_directory="surya_output"):
    """
    Automates running the 'surya_ocr' command-line tool for a PDF.

    Args:
        pdf_path (str): The full path to the PDF file to be processed.
        languages (list): A list of language codes (e.g., ["en", "si"]).
        output_directory (str): The directory where Surya will save its results.json.

    Returns:
        dict or None: A dictionary containing the parsed JSON results, or None if the command fails.
    """
    # Ensure the output directory exists
    os.makedirs(output_directory, exist_ok=True)

    # Construct the command as a list of strings for safety
    command = [
        "surya_ocr",
        pdf_path,
        "--output_dir", output_directory,
        # "--langs", ",".join(languages) # Join languages with commas for the --langs argument
    ]

    print(f"Executing command: {' '.join(command)}")

    try:
        # Run the command
        # capture_output=True: captures stdout and stderr
        # text=True: decodes stdout/stderr as text (otherwise they are bytes)
        # check=True: raises CalledProcessError if the command returns a non-zero exit code
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=True
        )

        print("Command executed successfully.")
        if result.stdout:
            print("STDOUT:")
            print(result.stdout)
        if result.stderr:
            print("STDERR:")
            print(result.stderr)

        # Surya's surya_ocr command typically creates a results.json file.
        # The filename inside results.json will be the base name of your input PDF.
        # So, we need to construct the expected path to the results file.
        # Surya stores results in a JSON where keys are input filenames.
        pdf_filename_without_ext = os.path.splitext(os.path.basename(pdf_path))[0]
        results_json_path = os.path.join(output_directory, "results.json")

        if os.path.exists(results_json_path):
            with open(results_json_path, 'r', encoding='utf-8') as f:
                full_results = json.load(f)
            
            # Surya's results.json contains a dictionary where keys are original filenames.
            # We want the results for our specific PDF.
            return full_results.get(pdf_filename_without_ext)
        else:
            print(f"Warning: 'results.json' not found at {results_json_path}. Check if Surya produced output.")
            return None

    except FileNotFoundError:
        print(f"Error: 'surya_ocr' command not found. Make sure Surya OCR is installed correctly and 'surya_ocr' is in your system's PATH.")
        print("You might need to activate your virtual environment or ensure 'surya_ocr' is executable.")
        return None
    except subprocess.CalledProcessError as e:
        print(f"Error running Surya OCR. Command failed with exit code {e.returncode}")
        print(f"STDOUT:\n{e.stdout}")
        print(f"STDERR:\n{e.stderr}")
        return None
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from '{results_json_path}'. The output might be malformed.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [8]:

if __name__ == "__main__":
    # Define your PDF path.
    # As per your saved information, your .arrow file (and likely your PDF)
    # is in a folder named 'data'.
    pdf_file_name = 'උම්මග්ග ජාතකය.pdf' # Replace with your actual PDF filename
    pdf_input_path = os.path.join('Diachronic Corpus', pdf_file_name)

    # Ensure the input PDF exists
    if not os.path.exists(pdf_input_path):
        print(f"Error: Input PDF not found at '{pdf_input_path}'. Please place your PDF in the 'data' folder.")
    else:
        # Define languages (e.g., Sinhala and English)
        ocr_languages = ["si", "en"]
        
        # Define output directory for Surya's results
        output_folder = os.path.join('data', 'surya_cli_results')

        # Run the automation
        print(f"Starting OCR for '{pdf_input_path}'...")
        surya_results = run_surya_ocr_cli(pdf_input_path, ocr_languages, output_folder)

        if surya_results:
            print("\n--- OCR Results ---")
            # You'll get a dictionary for the specific file
            print(json.dumps(surya_results, indent=4, ensure_ascii=False))

            # You can also access specific parts, e.g., the text lines of the first page
            if "pages" in surya_results and len(surya_results["pages"]) > 0:
                first_page_text_lines = surya_results["pages"][0].get("text_lines", [])
                print(f"\nText lines from first page ({len(first_page_text_lines)} lines):")
                for line in first_page_text_lines:
                    print(f"- {line.get('text', '')}")
        else:
            print("OCR process did not return valid results.")

Starting OCR for 'Diachronic Corpus/උම්මග්ග ජාතකය.pdf'...
Executing command: surya_ocr Diachronic Corpus/උම්මග්ග ජාතකය.pdf --output_dir data/surya_cli_results
Error running Surya OCR. Command failed with exit code -11
STDOUT:

STDERR:


















































































































































































































































































































































































































2025-10-03 12:20:35,144 [ERROR] surya: Error downloading model from text_recognition/2025_09_23. Attempt 1 of 3. Error: ('Connection broken: IncompleteRead(366230592 bytes read, 1072651902 more expected)', IncompleteRead(366230592 bytes read, 1072651902 more expected))
2025-10-03 12:20:35,144 [INFO] surya: Retrying in 5 seconds...
































In [51]:
file_path = "data/surya_cli_results/අජූද නීතිය /results.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
import json
import os
import re

# --- CONFIGURE YOUR FILE PATHS HERE ---
# Make sure 'pdf_file_name_for_key' exactly matches the basename Surya used
# to create the subfolder and the top-level key in results.json.
# From our previous debugging, this was 'අජූද නීතිය .pdf' which results in key 'අජූද නීතිය '.
pdf_file_name_for_key = 'අජූද නීතිය .pdf' # The original PDF filename you used for Surya CLI

# This constructs the path to the specific results.json file
pdf_basename_without_ext = os.path.splitext(os.path.basename(pdf_file_name_for_key))[0]
results_json_dir = os.path.join('data', 'surya_cli_results', pdf_basename_without_ext)
results_json_path = os.path.join(results_json_dir, 'results.json')

# Path to save the final concatenated text
output_text_dir = os.path.join('data', 'processed_text_output')
output_txt_path = os.path.join(output_text_dir, 'concatenated_document_simple.txt')

# --- CORE LOGIC STARTS HERE ---

print(f"Attempting to process results from: '{results_json_path}'")

if not os.path.exists(results_json_path):
    print(f"Error: 'results.json' not found at '{results_json_path}'. Check your path and that Surya CLI ran.")
else:
    try:
        # Read the entire JSON file content as a single string
        with open(results_json_path, 'r', encoding='utf-8') as f:
            json_string_content = f.read()

        # Regex to find all text values (handles escaped quotes)
        text_values_raw = re.findall(r'"text":\s*"((?:[^"\\]|\\.)*)"', json_string_content)

        concatenated_cleaned_text = []
        
        # Regex to remove HTML/LaTeX-like tags (e.g., <math>, <b>, <br>)
        tag_cleaner_re = re.compile(r'<[^>]+>')

        for raw_line_text in text_values_raw:
            # Clean out tags and strip remaining whitespace
            cleaned_line = tag_cleaner_re.sub('', raw_line_text)
            cleaned_line = cleaned_line.strip()

            if cleaned_line: # Only add if the line has content after cleaning
                concatenated_cleaned_text.append(cleaned_line)

        final_concatenated_output = "\n\n".join(concatenated_cleaned_text) # Separate pages by double newline

        # --- Output and Save ---
        print("\n--- Extracted Document Content ---")
        if final_concatenated_output:
            print(final_concatenated_output)
        else:
            print("[No readable text found after processing. Check results.json content.]")

        os.makedirs(output_text_dir, exist_ok=True) # Ensure output directory exists
        with open(output_txt_path, 'w', encoding='utf-8') as f:
            f.write(final_concatenated_output)
        print(f"\nConcatenated text saved to: {output_txt_path}")

    except json.JSONDecodeError as e:
        print(f"Error: Could not decode JSON from '{results_json_path}'. File might be corrupted or invalid JSON.")
        print(f"Details: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")